## Model Selection v3: Information-Theoretic Compression, Dual Interpolated Trigram LM, GPU GBDT Triad & Domain-Balanced Transductive Self-Training
**Cornell Tech CS 5785 Applied Machine Learning — Kaggle Competition (*Human or AI?*)**  
**Branch**: `model-selection-v2` (Targeting `0.97446` $\to$ `0.985+ / 0.99+` Kaggle Leaderboard AUROC)

---

## Executive Summary & Architecture Progression
- **Submission v1 (`0.93948` Public LB)**: Purely formatting-free OOF Stacker (`unlabeled.csv` SSL only).
- **Submission v2 (`0.97446` Public LB, $+0.0350$ jump)**: Introduced the **Topic-Masked Syntactic Skeleton (`View 4`)**, **Damped Structural + Stylometric LightGBM (`colsample_bytree=0.45`)**, and **Convex Logit Fusion** (`0.9517` $\to$ `0.9906` Unseen-Genre OOD AUROC).

### Four Orthogonal Upgrades in v3 (`submissions/team-35/plum/model_selection_v3_high_score.ipynb`)
To push our Kaggle score from **`0.97446`** toward the **`0.99+`** ceiling, we introduce four domain-invariant upgrades:
1. **Information-Theoretic & Discourse Cohesion Features (`13 New Features`)**:
   - **Zlib LZ77 Compression Ratios**: `comp_ratio_raw`, `comp_ratio_norm` (whitespace-normalized), `comp_h2_minus_h1` (second-half vs. first-half compression drift), and `comp_inc_h2` (conditional cross-compression of the second half given the first half, measuring long-range LLM self-similarity).
   - **Character Shannon & Conditional Bigram Entropy**: `char_entropy` and `cond_char_entropy` ($H(c_t | c_{t-1})$).
   - **Sentence-to-Sentence Lexical Cohesion & Line Rhythm**: `adj_sent_jaccard_mean`, `adj_sent_jaccard_max`, `sent_len_diff_mean`, `sent_starter_ttr` (diversity of sentence-initial words), `line_len_cv`, `line_end_punct_ratio` ($0.7635$ 1D Tree AUROC), and `line_start_cap_ratio`.
2. **5-Fold Out-of-Fold Dual Interpolated Trigram Language Model Log-Likelihood Ratios (`11 Token-Surprise Features`)**:
   - We fit generative Jelinek-Mercer interpolated $(1,2,3)$-gram Language Models separately on Human ($P_H(w_t|w_{t-2},w_{t-1})$) and AI ($P_{AI}(w_t|w_{t-2},w_{t-1})$) training folds and compute **token-by-token log-likelihood ratio moments** (`lm_llr_tri_mean`, `lm_llr_tri_std`, `lm_llr_tri_q75` — which alone achieves **$0.8860$ AUROC on completely unseen Novel & Film genres!**).
3. **GPU-Accelerated Multi-Family GBDT Triad (`LightGBM` + `CatBoost GPU` + `XGBoost CUDA`)**:
   - Instead of relying on a single tree algorithm, we train **LightGBM** (leaf-wise trees), **CatBoost GPU** (symmetric oblivious trees, achieving **$0.9884$ OOD AUROC** on unseen genres), and **XGBoost CUDA** (depth-wise histogram trees) across both the **115-feature Formatting-Free Pure+Info+LM+GenreZ view** and the **130-feature Damped-Structural+Info+LM+GenreZ view**.
4. **Domain-Balanced Transductive Self-Training Across `unlabeled.csv` AND `test.csv`**:
   - Because `test.csv` contains $432$ Poetry/Verse and $418$ Book/Novel Summary passages that are rare in `train.csv`, our Stage-1 Ensemble selects the **highest-confidence balanced AI and Human examples within every genre of `test.csv` and `unlabeled.csv`** ($|\hat{P} - 0.5| \ge 0.45$) so Stage-2 linear and GBDT models directly adapt to the vocabulary and stanza structure of `test.csv`.

In [1]:
import time
import math
import zlib
import re
import warnings
from collections import Counter
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from scipy.special import logit, expit
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display

warnings.filterwarnings('ignore')
sns.set_theme(
    style='whitegrid',
    palette='deep',
    font_scale=0.80,
    rc={
        'figure.dpi': 95,
        'savefig.dpi': 115,
        'font.size': 8.0,
        'axes.titlesize': 8.5,
        'axes.labelsize': 8.0,
        'xtick.labelsize': 7.5,
        'ytick.labelsize': 7.5,
        'legend.fontsize': 7.5,
        'figure.titlesize': 9.5,
    }
)
display(HTML(
    "<style>"
    ".jp-RenderedHTMLCommon, .rendered_html { font-size: 12.5px !important; line-height: 1.45 !important; }"
    ".jp-RenderedHTMLCommon table, .rendered_html table, table.dataframe { font-size: 11px !important; }"
    ".jp-RenderedHTMLCommon h1, .rendered_html h1 { font-size: 1.35em !important; }"
    ".jp-RenderedHTMLCommon h2, .rendered_html h2 { font-size: 1.18em !important; }"
    ".jp-RenderedHTMLCommon h3, .rendered_html h3 { font-size: 1.05em !important; }"
    ".jp-RenderedHTMLCommon h4, .rendered_html h4 { font-size: 0.98em !important; }"
    "</style>"
))

# Load model metadata and hyperparameters from external YAML
def load_config(rel_path):
    candidates = [
        Path(rel_path),
        Path('models') / Path(rel_path).name,
        Path('submissions/team-35/plum') / rel_path,
        Path('submissions/plum') / rel_path,
    ]
    for p in candidates:
        if p.exists():
            with open(p, 'r', encoding='utf-8') as f:
                return yaml.safe_load(f)
    raise FileNotFoundError(f"Config file not found in any candidate path: {candidates}")

cfg = load_config('models/model_selection_v3_high_score.yaml')
RANDOM_SEED = cfg.get('random_seed', 42)
np.random.seed(RANDOM_SEED)

DATA_DIR = Path('../../../data') if Path('../../../data/train.csv').exists() else (Path('../data') if Path('../data/train.csv').exists() else Path('data'))
train_df = pd.read_csv(DATA_DIR / 'train.csv')
unlabeled_df = pd.read_csv(DATA_DIR / 'unlabeled.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
y_all = train_df['label'].values
print(f"Loaded train: {train_df.shape}, unlabeled: {unlabeled_df.shape}, test: {test_df.shape}")


Loaded train: (10000, 3), unlabeled: (3000, 2), test: (3000, 2)


In [2]:
# 1. Core Stylometric, Information-Theoretic Compression & Dual Interpolated Language Model Feature Extractors
STOPWORDS = {
    'the','be','to','of','and','a','in','that','have','i','it','for','not','on','with',
    'he','as','you','do','at','this','but','his','by','from','they','we','say','her',
    'she','or','an','will','my','one','all','would','there','their','what','so','up',
    'out','if','about','who','get','which','go','me','when','make','can','like','time',
    'no','just','him','know','take','people','into','year','your','good','some','could',
    'them','see','other','than','then','now','look','only','come','its','over','think',
    'also','back','after','use','two','how','our','work','first','well','way','even',
    'new','want','because','any','these','give','day','most','us','is','are','was','were',
    'been','has','had','did','does','am','may','might','must','shall','should','ought',
    'need','dare','used'
}
PRONOUNS_1P = {'i','me','my','mine','myself','we','us','our','ours','ourselves'}
PRONOUNS_2P = {'you','your','yours','yourself','yourselves'}
PRONOUNS_3P = {'he','him','his','himself','she','her','hers','herself','it','its','itself','they','them','their','theirs','themselves'}
DET_ARTICLES = {'the','a','an','this','that','these','those','each','every','either','neither','some','any','no','much','many','few','several','all','both','half'}
PREPOSITIONS = {'of','in','to','for','with','on','at','from','by','about','as','into','like','through','after','over','between','out','against','during','without','before','under','around','among','within','upon','toward','towards','throughout','despite','beyond','across','behind','above','near','along','until','via'}
AUX_MODALS = {'be','am','is','are','was','were','been','being','have','has','had','having','do','does','did','will','would','shall','should','can','could','may','might','must','ought'}
CONJUNCTIONS = {'and','but','or','nor','for','yet','so','although','because','since','unless','while','whereas','whether','though','if','when','whenever','where','wherever','before','after','until','once'}
AI_TRANSITIONS = [
    'moreover','furthermore','in conclusion','delve','crucial','notably','additionally',
    'consequently','nevertheless','nonetheless','in summary','overall','importantly',
    'specifically','ultimately','vital','essential','significant','comprehensive',
    'robust','enhance','facilitate','utilize','leverage','paradigm','realm','tapestry',
    'testament','underscore','pivotal','intricate','foster','embark','nuanced','multifaceted'
]
HEDGING_WORDS = {
    'perhaps','maybe','possibly','probably','apparently','seemingly','arguably','roughly',
    'approximately','generally','typically','usually','often','sometimes','likely','unlikely',
    'suggest','suggests','indicate','indicates','appear','appears','seem','seems'
}
FUNCTION_VOCAB = (
    STOPWORDS | PRONOUNS_1P | PRONOUNS_2P | PRONOUNS_3P | DET_ARTICLES |
    PREPOSITIONS | AUX_MODALS | CONJUNCTIONS | HEDGING_WORDS |
    set(w for m in AI_TRANSITIONS for w in m.split())
)

SYL_RE_1 = re.compile(r'(?:[^laeiouy]es|ed|[^laeiouy]e)$')
SYL_RE_2 = re.compile(r'^y')
SYL_RE_3 = re.compile(r'[aeiouy]{1,2}')
TOKEN_RE = re.compile(r"[a-zA-Z]+(?:'[a-zA-Z]+)?")
TOKEN_PUNCT_RE = re.compile(r"[a-zA-Z]+(?:'[a-zA-Z]+)?|[0-9]+|[^\s\w]")
SENT_RE = re.compile(r'[.!?]+(?:\s+|$)')
CURLY_RE = re.compile(r'[\u201c\u201d\u2018\u2019]')
BULLET_RE = re.compile(r'(?:^|\n)\s*[-*•]\s')
HTML_BR_RE = re.compile(r'<br\s*/?>', re.IGNORECASE)

def strip_formatting(text: str) -> str:
    text = HTML_BR_RE.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def mask_content_words(text: str) -> str:
    text = HTML_BR_RE.sub(' \n ', text)
    tokens = TOKEN_PUNCT_RE.findall(text)
    out = []
    for tok in tokens:
        low = tok.lower()
        if low in FUNCTION_VOCAB:
            out.append(low)
        elif tok.isdigit():
            out.append('_NUM_')
        elif not tok[0].isalpha():
            out.append(tok)
        else:
            cap = 'T' if tok[0].isupper() else 'w'
            if len(low) >= 7:
                out.append(f'_{cap}_{low[-3:]}')
            elif len(low) >= 4:
                out.append(f'_{cap}_{low[-2:]}')
            else:
                out.append(f'_{cap}_s')
    return ' '.join(out)

def classify_observable_domain(text: str) -> str:
    t_low = text.lower()
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    avg_line_words = len(text.split()) / max(len(lines), 1)
    if re.search(r'\b(ingredients|tablespoon|teaspoon|preheat|oven|chopped|cup|ounce)\b', t_low):
        return '1. Recipe / Cooking'
    if re.search(r'\b(we propose|this paper|state-of-the-art|neural network|convolutional|segmentation|algorithm|dataset|experiments|abstract|framework|method|model|learning)\b', t_low):
        return '2. Academic / ArXiv Abstract'
    if len(lines) >= 8 and avg_line_words < 10 and sum(1 for l in lines[:8] if l and l[0].isupper()) >= 6:
        return '3. Poetry / Verse / Lyrics'
    if re.search(r'\b(the novel|the book|the story|protagonist|narrates|published in|chapter)\b', t_low):
        return '4. Book / Novel Summary'
    if re.search(r'\b(the film|the movie|directed by|starring|box office|episode|season)\b', t_low) or '<br' in t_low:
        return '5. Film / TV / Review'
    if re.search(r'\b(born|politician|member of the|census-designated|is a|was a|located in)\b', t_low) and any(c.isdigit() for c in text[:150]):
        return '6. Encyclopedia / Wiki Bio'
    if re.search(r"\b(i am|i have|my|i'm|i was|reddit|tldr|edit:)\b", t_low):
        return '7. First-Person / Social'
    return '8. General / News / Other'

def count_syllables(word: str) -> int:
    if len(word) <= 3:
        return 1
    word = SYL_RE_1.sub('', word)
    word = SYL_RE_2.sub('', word)
    syls = SYL_RE_3.findall(word)
    return max(1, len(syls))

def extract_all_base_and_info_features(df: pd.DataFrame) -> pd.DataFrame:
    feats = []
    for text in df['text']:
        char_count = len(text)
        words_raw = text.split()
        word_count = max(len(words_raw), 1)
        alpha_tokens = TOKEN_RE.findall(text)
        lower_tokens = [w.lower() for w in alpha_tokens]
        n_alpha = max(len(lower_tokens), 1)

        sents = [s.strip() for s in SENT_RE.split(text) if s.strip()]
        sent_count = max(len(sents), 1)
        sent_lens = [len(s.split()) for s in sents] if sents else [word_count]
        avg_sent_len = float(np.mean(sent_lens))
        std_sent_len = float(np.std(sent_lens)) if len(sent_lens) > 1 else 0.0
        cv_sent_len = std_sent_len / max(avg_sent_len, 1.0)
        max_sent_len = float(np.max(sent_lens))
        min_sent_len = float(np.min(sent_lens))

        lines = text.split('\n')
        line_count = len(lines)
        non_empty_lines = [l.strip() for l in lines if l.strip()]
        non_empty_line_count = max(len(non_empty_lines), 1)
        newline_count = text.count('\n')
        double_newline_count = text.count('\n\n')
        has_double_newline = 1.0 if '\n\n' in text else 0.0
        has_single_newline_only = 1.0 if ('\n' in text and '\n\n' not in text) else 0.0
        single_newline_count = max(0, newline_count - 2 * double_newline_count)
        avg_line_len_words = word_count / non_empty_line_count

        vocab = {}
        for w in lower_tokens:
            vocab[w] = vocab.get(w, 0) + 1
        n_unique = len(vocab)
        ttr = n_unique / n_alpha
        root_ttr = n_unique / np.sqrt(n_alpha)
        log_ttr = np.log(max(n_unique, 1)) / max(np.log(max(n_alpha, 2)), 1e-5)
        hapax = sum(1 for v in vocab.values() if v == 1)
        dis_legomena = sum(1 for v in vocab.values() if v == 2)
        hapax_ratio = hapax / n_alpha
        hapax_vocab_ratio = hapax / max(n_unique, 1)
        dis_legomena_ratio = dis_legomena / n_alpha
        m1 = n_alpha
        m2 = sum(v * v for v in vocab.values())
        yules_k = 10000.0 * (m2 - m1) / max(m1 * m1, 1)

        word_lens = [len(w) for w in lower_tokens] if lower_tokens else [1]
        avg_word_len = float(np.mean(word_lens))
        std_word_len = float(np.std(word_lens))
        long_word_ratio = sum(1 for l in word_lens if l >= 7) / n_alpha
        short_word_ratio = sum(1 for l in word_lens if l <= 3) / n_alpha

        if len(lower_tokens) >= 3:
            bgs = [(lower_tokens[i], lower_tokens[i+1]) for i in range(len(lower_tokens)-1)]
            tgs = [(lower_tokens[i], lower_tokens[i+1], lower_tokens[i+2]) for i in range(len(lower_tokens)-2)]
            rep_2gram = 1.0 - len(set(bgs)) / len(bgs)
            rep_3gram = 1.0 - len(set(tgs)) / len(tgs)
        else:
            rep_2gram = 0.0
            rep_3gram = 0.0

        scale_c = 1000.0 / max(char_count, 1)
        comma_rate = text.count(',') * scale_c
        period_rate = text.count('.') * scale_c
        semicolon_rate = text.count(';') * scale_c
        colon_rate = text.count(':') * scale_c
        exclamation_rate = text.count('!') * scale_c
        question_rate = text.count('?') * scale_c
        hyphen_rate = text.count('-') * scale_c
        en_dash_rate = text.count('\u2013') * scale_c
        em_dash_rate = text.count('\u2014') * scale_c
        straight_quote_rate = text.count('"') * scale_c
        single_quote_rate = text.count("'") * scale_c
        curly_quote_rate = len(CURLY_RE.findall(text)) * scale_c
        paren_rate = (text.count('(') + text.count(')')) * scale_c
        slash_rate = text.count('/') * scale_c

        starts_with_quote = 1.0 if text.startswith('"') else 0.0
        ends_with_quote = 1.0 if text.endswith('"') else 0.0
        double_space_rate = text.count('  ') * scale_c
        has_bullet_list = 1.0 if BULLET_RE.search(text) else 0.0
        has_html_br = 1.0 if HTML_BR_RE.search(text) else 0.0

        upper_chars = sum(1 for c in text if c.isupper())
        digit_chars = sum(1 for c in text if c.isdigit())
        space_chars = sum(1 for c in text if c.isspace())
        punct_chars = sum(1 for c in text if not c.isalnum() and not c.isspace())
        upper_char_ratio = upper_chars / max(char_count, 1)
        digit_char_ratio = digit_chars / max(char_count, 1)
        space_char_ratio = space_chars / max(char_count, 1)
        punct_char_ratio = punct_chars / max(char_count, 1)
        title_word_ratio = sum(1 for w in alpha_tokens if w.istitle()) / n_alpha
        allcaps_word_ratio = sum(1 for w in alpha_tokens if w.isupper() and len(w) > 1) / n_alpha

        syl_counts = [count_syllables(w) for w in lower_tokens] if lower_tokens else [1]
        total_syls = sum(syl_counts)
        complex_words = sum(1 for s in syl_counts if s >= 3)
        avg_syl_per_word = total_syls / n_alpha
        complex_word_ratio = complex_words / n_alpha
        fk_grade = 0.39 * avg_sent_len + 11.8 * avg_syl_per_word - 15.59
        flesch_ease = 206.835 - 1.015 * avg_sent_len - 84.6 * avg_syl_per_word
        gunning_fog = 0.4 * (avg_sent_len + 100.0 * complex_word_ratio)
        ari = 4.71 * (char_count / word_count) + 0.5 * avg_sent_len - 21.43
        coleman_liau = 0.0588 * (sum(word_lens) / word_count * 100.0) - 0.296 * (sent_count / word_count * 100.0) - 15.8

        stop_ratio = sum(1 for w in lower_tokens if w in STOPWORDS) / n_alpha
        pron1_ratio = sum(1 for w in lower_tokens if w in PRONOUNS_1P) / n_alpha
        pron2_ratio = sum(1 for w in lower_tokens if w in PRONOUNS_2P) / n_alpha
        pron3_ratio = sum(1 for w in lower_tokens if w in PRONOUNS_3P) / n_alpha
        pron_total_ratio = pron1_ratio + pron2_ratio + pron3_ratio
        det_ratio = sum(1 for w in lower_tokens if w in DET_ARTICLES) / n_alpha
        prep_ratio = sum(1 for w in lower_tokens if w in PREPOSITIONS) / n_alpha
        aux_ratio = sum(1 for w in lower_tokens if w in AUX_MODALS) / n_alpha
        conj_ratio = sum(1 for w in lower_tokens if w in CONJUNCTIONS) / n_alpha

        adv_ly_ratio = sum(1 for w in lower_tokens if len(w) > 4 and w.endswith('ly')) / n_alpha
        verb_ing_ratio = sum(1 for w in lower_tokens if len(w) > 4 and w.endswith('ing')) / n_alpha
        verb_ed_ratio = sum(1 for w in lower_tokens if len(w) > 4 and w.endswith('ed')) / n_alpha
        noun_morph_ratio = sum(1 for w in lower_tokens if len(w) > 5 and w.endswith(('tion','sion','ment','ness','ity','ance','ence','ship','ism'))) / n_alpha
        adj_morph_ratio = sum(1 for w in lower_tokens if len(w) > 5 and w.endswith(('able','ible','al','ful','ic','ive','less','ous'))) / n_alpha
        contraction_rate = sum(1 for w in lower_tokens if "'" in w) / n_alpha

        text_lower = text.lower()
        ai_trans_count = sum(text_lower.count(marker) for marker in AI_TRANSITIONS)
        ai_trans_rate = ai_trans_count / word_count * 100.0
        hedge_rate = sum(1 for w in lower_tokens if w in HEDGING_WORDS) / n_alpha * 100.0

        # Information-Theoretic Compression & Entropy Features
        raw_bytes = text.encode('utf-8')
        n_bytes = max(len(raw_bytes), 1)
        comp_raw = len(zlib.compress(raw_bytes, level=9))
        comp_ratio_raw = comp_raw / n_bytes

        norm_text = strip_formatting(text)
        norm_bytes = norm_text.encode('utf-8')
        n_norm_bytes = max(len(norm_bytes), 1)
        comp_norm = len(zlib.compress(norm_bytes, level=9))
        comp_ratio_norm = comp_norm / n_norm_bytes

        mid = len(norm_bytes) // 2
        if mid > 20:
            c_h1 = len(zlib.compress(norm_bytes[:mid], level=9)) / mid
            c_h2 = len(zlib.compress(norm_bytes[mid:], level=9)) / max(len(norm_bytes) - mid, 1)
            c_inc_h2 = (comp_norm - len(zlib.compress(norm_bytes[:mid], level=9))) / max(len(norm_bytes) - mid, 1)
        else:
            c_h1 = c_h2 = c_inc_h2 = comp_ratio_norm

        c_counts = Counter(norm_text)
        n_c = max(len(norm_text), 1)
        char_entropy = -sum((v / n_c) * math.log2(v / n_c) for v in c_counts.values())
        if len(norm_text) >= 2:
            bg_counts = Counter(zip(norm_text[:-1], norm_text[1:]))
            n_bg = len(norm_text) - 1
            bg_entropy = -sum((v / n_bg) * math.log2(v / n_bg) for v in bg_counts.values())
            cond_char_entropy = bg_entropy - char_entropy
        else:
            cond_char_entropy = 0.0

        sents_norm = [s.strip() for s in SENT_RE.split(norm_text) if s.strip()]
        if len(sents_norm) >= 2:
            sent_sets = [set(re.findall(r'[a-zA-Z]+', s.lower())) for s in sents_norm]
            jaccards = [
                len(sent_sets[i] & sent_sets[i+1]) / max(len(sent_sets[i] | sent_sets[i+1]), 1)
                for i in range(len(sent_sets) - 1)
            ]
            adj_sent_jaccard_mean = float(np.mean(jaccards))
            adj_sent_jaccard_max = float(np.max(jaccards))
            sent_lens_n = [len(s.split()) for s in sents_norm]
            sent_len_diff_mean = float(np.mean(np.abs(np.diff(sent_lens_n)))) / max(np.mean(sent_lens_n), 1.0)
            first_words = [s.split()[0].lower() for s in sents_norm if s.split()]
            sent_starter_ttr = len(set(first_words)) / max(len(first_words), 1)
        else:
            adj_sent_jaccard_mean = 0.0
            adj_sent_jaccard_max = 0.0
            sent_len_diff_mean = 0.0
            sent_starter_ttr = 1.0

        if len(non_empty_lines) >= 2:
            line_lens = [len(l) for l in non_empty_lines]
            line_len_cv = float(np.std(line_lens)) / max(np.mean(line_lens), 1.0)
            line_end_punct_ratio = sum(1 for l in non_empty_lines if l[-1] in '.!?:;"\'\u201d\u2019)') / len(non_empty_lines)
            line_start_cap_ratio = sum(1 for l in non_empty_lines if l[0].isupper() or l[0] in '"-\u201c*•0123456789') / len(non_empty_lines)
        else:
            line_len_cv = 0.0
            line_end_punct_ratio = 1.0
            line_start_cap_ratio = 1.0

        feats.append({
            'word_count': word_count, 'char_count': char_count, 'sent_count': sent_count,
            'avg_sent_len': avg_sent_len, 'std_sent_len': std_sent_len, 'cv_sent_len': cv_sent_len,
            'max_sent_len': max_sent_len, 'min_sent_len': min_sent_len,
            'line_count': line_count, 'newline_count': newline_count,
            'double_newline_count': double_newline_count, 'single_newline_count': single_newline_count,
            'has_double_newline': has_double_newline, 'has_single_newline_only': has_single_newline_only,
            'avg_line_len_words': avg_line_len_words,
            'ttr': ttr, 'root_ttr': root_ttr, 'log_ttr': log_ttr,
            'hapax_ratio': hapax_ratio, 'hapax_vocab_ratio': hapax_vocab_ratio,
            'dis_legomena_ratio': dis_legomena_ratio, 'yules_k': yules_k,
            'avg_word_len': avg_word_len, 'std_word_len': std_word_len,
            'long_word_ratio': long_word_ratio, 'short_word_ratio': short_word_ratio,
            'rep_2gram': rep_2gram, 'rep_3gram': rep_3gram,
            'comma_rate': comma_rate, 'period_rate': period_rate,
            'semicolon_rate': semicolon_rate, 'colon_rate': colon_rate,
            'exclamation_rate': exclamation_rate, 'question_rate': question_rate,
            'hyphen_rate': hyphen_rate, 'en_dash_rate': en_dash_rate,
            'em_dash_rate': em_dash_rate, 'straight_quote_rate': straight_quote_rate,
            'single_quote_rate': single_quote_rate, 'curly_quote_rate': curly_quote_rate,
            'paren_rate': paren_rate, 'slash_rate': slash_rate,
            'starts_with_quote': starts_with_quote, 'ends_with_quote': ends_with_quote,
            'double_space_rate': double_space_rate, 'has_bullet_list': has_bullet_list,
            'has_html_br': has_html_br,
            'upper_char_ratio': upper_char_ratio, 'digit_char_ratio': digit_char_ratio,
            'space_char_ratio': space_char_ratio, 'punct_char_ratio': punct_char_ratio,
            'title_word_ratio': title_word_ratio, 'allcaps_word_ratio': allcaps_word_ratio,
            'avg_syl_per_word': avg_syl_per_word, 'complex_word_ratio': complex_word_ratio,
            'fk_grade': fk_grade, 'flesch_ease': flesch_ease,
            'gunning_fog': gunning_fog, 'ari': ari, 'coleman_liau': coleman_liau,
            'stop_ratio': stop_ratio, 'pron1_ratio': pron1_ratio,
            'pron2_ratio': pron2_ratio, 'pron3_ratio': pron3_ratio,
            'pron_total_ratio': pron_total_ratio, 'det_ratio': det_ratio,
            'prep_ratio': prep_ratio, 'aux_ratio': aux_ratio, 'conj_ratio': conj_ratio,
            'adv_ly_ratio': adv_ly_ratio, 'verb_ing_ratio': verb_ing_ratio,
            'verb_ed_ratio': verb_ed_ratio, 'noun_morph_ratio': noun_morph_ratio,
            'adj_morph_ratio': adj_morph_ratio, 'contraction_rate': contraction_rate,
            'ai_trans_rate': ai_trans_rate, 'hedge_rate': hedge_rate,
            'comp_ratio_raw': comp_ratio_raw, 'comp_ratio_norm': comp_ratio_norm,
            'comp_h2_minus_h1': c_h2 - c_h1, 'comp_inc_h2': c_inc_h2,
            'char_entropy': char_entropy, 'cond_char_entropy': cond_char_entropy,
            'adj_sent_jaccard_mean': adj_sent_jaccard_mean, 'adj_sent_jaccard_max': adj_sent_jaccard_max,
            'sent_len_diff_mean': sent_len_diff_mean, 'sent_starter_ttr': sent_starter_ttr,
            'line_len_cv': line_len_cv, 'line_end_punct_ratio': line_end_punct_ratio,
            'line_start_cap_ratio': line_start_cap_ratio
        })
    return pd.DataFrame(feats, index=df.index)

In [3]:
# 2. Dual Interpolated Trigram Language Model Log-Likelihood Ratio Extractor (5-Fold OOF on Train)
class InterpolatedNGramLM:
    def __init__(self, alpha=None):
        if alpha is None:
            alpha = cfg.get('language_model', {}).get('alpha', 0.5)
        self.alpha = alpha
        self.uni = Counter()
        self.bi = Counter()
        self.tri = Counter()
        self.total = 0
        self.V = 1

    def fit(self, texts):
        for text in texts:
            toks = ['<BOS>', '<BOS>'] + [t.lower() for t in TOKEN_PUNCT_RE.findall(text)] + ['<EOS>']
            self.total += len(toks) - 2
            for i in range(2, len(toks)):
                w0, w1, w2 = toks[i-2], toks[i-1], toks[i]
                self.uni[w2] += 1
                self.bi[(w1, w2)] += 1
                self.tri[(w0, w1, w2)] += 1
        self.V = max(len(self.uni), 1)
        return self

    def log_probs(self, text):
        toks = ['<BOS>', '<BOS>'] + [t.lower() for t in TOKEN_PUNCT_RE.findall(text)] + ['<EOS>']
        ll1, ll2, ll3 = [], [], []
        denom = self.total + self.alpha * self.V
        for i in range(2, len(toks)):
            w0, w1, w2 = toks[i-2], toks[i-1], toks[i]
            p1 = (self.uni[w2] + self.alpha) / denom
            c1 = self.uni[w1]
            p2 = 0.72 * (self.bi[(w1, w2)] / c1) + 0.28 * p1 if c1 > 0 else p1
            c01 = self.bi[(w0, w1)]
            p3 = 0.62 * (self.tri[(w0, w1, w2)] / c01) + 0.38 * p2 if c01 > 0 else p2
            ll1.append(math.log(p1))
            ll2.append(math.log(p2))
            ll3.append(math.log(p3))
        return np.array(ll1), np.array(ll2), np.array(ll3)

def score_lm_pair(text, lm_h, lm_a):
    u_h, b_h, t_h = lm_h.log_probs(text)
    u_a, b_a, t_a = lm_a.log_probs(text)
    llr_u = u_a - u_h
    llr_b = b_a - b_h
    llr_t = t_a - t_h
    return {
        'lm_llr_uni_mean': float(np.mean(llr_u)),
        'lm_llr_bi_mean': float(np.mean(llr_b)),
        'lm_llr_tri_mean': float(np.mean(llr_t)),
        'lm_llr_tri_std': float(np.std(llr_t)),
        'lm_llr_tri_q75': float(np.quantile(llr_t, 0.75)),
        'lm_llr_tri_q25': float(np.quantile(llr_t, 0.25)),
        'lm_human_tri_ll_mean': float(np.mean(t_h)),
        'lm_human_tri_ll_std': float(np.std(t_h)),
        'lm_ai_tri_ll_mean': float(np.mean(t_a)),
        'lm_ai_tri_ll_std': float(np.std(t_a)),
        'lm_surprise_diff_std': float(np.std(t_a) - np.std(t_h)),
    }

t0 = time.time()
for df_p in [train_df, unlabeled_df, test_df]:
    df_p['domain'] = df_p['text'].apply(classify_observable_domain)

X_base_tr = extract_all_base_and_info_features(train_df)
X_base_un = extract_all_base_and_info_features(unlabeled_df)
X_base_te = extract_all_base_and_info_features(test_df)

# Extract 5-Fold OOF Dual Language Model features on train_df, and full-fit features on unlabeled_df and test_df
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
lm_oof_rows = [None] * len(train_df)
for t_idx, v_idx in skf.split(train_df, y_all):
    lm_h = InterpolatedNGramLM().fit(train_df.iloc[t_idx].loc[y_all[t_idx] == 0, 'text'])
    lm_a = InterpolatedNGramLM().fit(train_df.iloc[t_idx].loc[y_all[t_idx] == 1, 'text'])
    for idx in v_idx:
        lm_oof_rows[idx] = score_lm_pair(train_df.loc[idx, 'text'], lm_h, lm_a)

lm_h_full = InterpolatedNGramLM().fit(train_df.loc[y_all == 0, 'text'])
lm_a_full = InterpolatedNGramLM().fit(train_df.loc[y_all == 1, 'text'])
lm_tr_df = pd.DataFrame(lm_oof_rows, index=train_df.index)
lm_un_df = pd.DataFrame([score_lm_pair(txt, lm_h_full, lm_a_full) for txt in unlabeled_df['text']], index=unlabeled_df.index)
lm_te_df = pd.DataFrame([score_lm_pair(txt, lm_h_full, lm_a_full) for txt in test_df['text']], index=test_df.index)

print(f"Extracted Base + Information-Theoretic + Dual Interpolated LM features in {time.time()-t0:.1f}s.")


Extracted Base + Information-Theoretic + Dual Interpolated LM features in 51.3s.


In [4]:
# 3. Assemble Complete Pure (115 cols) and Damped-Structural (131 cols) Feature Matrices
raw_fmt_cols = [
    'double_newline_count', 'has_double_newline', 'has_single_newline_only',
    'single_newline_count', 'newline_count', 'line_count', 'avg_line_len_words',
    'starts_with_quote', 'ends_with_quote', 'double_space_rate', 'has_bullet_list', 'has_html_br',
    'comp_ratio_raw', 'line_len_cv', 'line_end_punct_ratio', 'line_start_cap_ratio'
]
pure_base_cols = [c for c in X_base_tr.columns if c not in raw_fmt_cols]

human_ref = X_base_tr.loc[y_all == 0]
for col in ['ttr', 'log_ttr', 'root_ttr', 'avg_sent_len', 'word_count', 'char_count', 'avg_word_len']:
    med = human_ref[col].median()
    iqr = max(human_ref[col].quantile(0.75) - human_ref[col].quantile(0.25), 1e-5)
    for df_s in [X_base_tr, X_base_un, X_base_te]:
        df_s[f'{col}_abs_dev'] = np.abs(df_s[col] - med) / iqr
    pure_base_cols.append(f'{col}_abs_dev')

genre_target_cols = [
    'log_ttr', 'ttr', 'root_ttr', 'yules_k', 'rep_3gram', 'rep_2gram',
    'avg_sent_len', 'std_sent_len', 'cv_sent_len', 'avg_word_len', 'coleman_liau',
    'comma_rate', 'period_rate', 'colon_rate', 'hyphen_rate', 'punct_char_ratio',
    'stop_ratio', 'prep_ratio', 'det_ratio', 'pron1_ratio', 'pron3_ratio', 'digit_char_ratio'
]
global_h_mean = X_base_tr.loc[y_all == 0, genre_target_cols].mean()
global_h_std = X_base_tr.loc[y_all == 0, genre_target_cols].std().clip(lower=1e-4)
domain_stats = {}
for dom in train_df['domain'].unique():
    mask_dom = (train_df['domain'].values == dom) & (y_all == 0)
    if mask_dom.sum() >= 10:
        domain_stats[dom] = (
            X_base_tr.loc[mask_dom, genre_target_cols].mean(),
            X_base_tr.loc[mask_dom, genre_target_cols].std().clip(lower=1e-4)
        )
    else:
        domain_stats[dom] = (global_h_mean, global_h_std)

def finalize_feature_table(base_df, dom_s, lm_df):
    z_list = []
    for idx, dom in enumerate(dom_s):
        m, s = domain_stats.get(dom, (global_h_mean, global_h_std))
        row_vals = base_df.iloc[idx][genre_target_cols]
        z_list.append(((row_vals - m) / s).values)
    z_cols = [f'{c}_genre_z' for c in genre_target_cols]
    z_df = pd.DataFrame(z_list, index=base_df.index, columns=z_cols)
    z_df['genre_z_l2_norm'] = np.sqrt(np.mean(np.square(z_df.values), axis=1))
    z_df['genre_z_max_abs'] = np.max(np.abs(z_df[z_cols].values), axis=1)
    out = pd.concat([base_df, z_df, lm_df], axis=1)
    for c in ['double_newline_count', 'single_newline_count', 'newline_count', 'line_count']:
        out[f'{c}_log'] = np.log1p(out[c])
    return out, z_cols + ['genre_z_l2_norm', 'genre_z_max_abs']

X_full_tr, z_cols = finalize_feature_table(X_base_tr, train_df['domain'], lm_tr_df)
X_full_un, _ = finalize_feature_table(X_base_un, unlabeled_df['domain'], lm_un_df)
X_full_te, _ = finalize_feature_table(X_base_te, test_df['domain'], lm_te_df)

pure_116_cols = pure_base_cols + z_cols + list(lm_tr_df.columns)
damped_fmt_cols = [
    'double_newline_count_log', 'single_newline_count_log', 'newline_count_log', 'line_count_log',
    'has_double_newline', 'has_single_newline_only', 'avg_line_len_words',
    'starts_with_quote', 'ends_with_quote', 'double_space_rate', 'has_bullet_list', 'has_html_br',
    'comp_ratio_raw', 'line_len_cv', 'line_end_punct_ratio', 'line_start_cap_ratio'
]
all_132_cols = pure_116_cols + damped_fmt_cols

print(f"Final Feature Views -> Pure+Info+LM+GenreZ: {len(pure_116_cols)} cols | Full Damped Structural+Info+LM+Stylo: {len(all_132_cols)} cols")

Final Feature Views -> Pure+Info+LM+GenreZ: 116 cols | Full Damped Structural+Info+LM+Stylo: 132 cols


In [5]:
# 4. Stage-1 5-Fold Cross-Validated Multi-Family Training (4-View N-Grams + GPU GBDT Triad)
t0 = time.time()
tr_clean = train_df['text'].apply(strip_formatting)
un_clean = unlabeled_df['text'].apply(strip_formatting)
te_clean = test_df['text'].apply(strip_formatting)

tr_masked = train_df['text'].apply(mask_content_words)
un_masked = unlabeled_df['text'].apply(mask_content_words)
te_masked = test_df['text'].apply(mask_content_words)

all_clean_corpus = pd.concat([tr_clean, un_clean, te_clean], ignore_index=True)
all_masked_corpus = pd.concat([tr_masked, un_masked, te_masked], ignore_index=True)

v_cfg = cfg['vectorizers']
vw_full = TfidfVectorizer(ngram_range=tuple(v_cfg['word_tfidf']['ngram_range']), min_df=v_cfg['word_tfidf']['min_df'], max_features=v_cfg['word_tfidf']['max_features'], sublinear_tf=v_cfg['word_tfidf']['sublinear_tf']).fit(all_clean_corpus)
vcwb_full = TfidfVectorizer(ngram_range=tuple(v_cfg['char_wb_tfidf']['ngram_range']), analyzer=v_cfg['char_wb_tfidf']['analyzer'], min_df=v_cfg['char_wb_tfidf']['min_df'], max_features=v_cfg['char_wb_tfidf']['max_features'], sublinear_tf=v_cfg['char_wb_tfidf']['sublinear_tf']).fit(all_clean_corpus)
vccr_full = TfidfVectorizer(ngram_range=tuple(v_cfg['char_cross_tfidf']['ngram_range']), analyzer=v_cfg['char_cross_tfidf']['analyzer'], min_df=v_cfg['char_cross_tfidf']['min_df'], max_features=v_cfg['char_cross_tfidf']['max_features'], sublinear_tf=v_cfg['char_cross_tfidf']['sublinear_tf']).fit(all_clean_corpus)
vmask_full = TfidfVectorizer(ngram_range=tuple(v_cfg['mask_tfidf']['ngram_range']), token_pattern=v_cfg['mask_tfidf']['token_pattern'], min_df=v_cfg['mask_tfidf']['min_df'], max_features=v_cfg['mask_tfidf']['max_features'], sublinear_tf=v_cfg['mask_tfidf']['sublinear_tf']).fit(all_masked_corpus)

Xw_tr, Xw_un, Xw_te = vw_full.transform(tr_clean), vw_full.transform(un_clean), vw_full.transform(te_clean)
Xcwb_tr, Xcwb_un, Xcwb_te = vcwb_full.transform(tr_clean), vcwb_full.transform(un_clean), vcwb_full.transform(te_clean)
Xccr_tr, Xccr_un, Xccr_te = vccr_full.transform(tr_clean), vccr_full.transform(un_clean), vccr_full.transform(te_clean)
Xm_tr, Xm_un, Xm_te = vmask_full.transform(tr_masked), vmask_full.transform(un_masked), vmask_full.transform(te_masked)

X_4v_tr = hstack([Xw_tr, Xcwb_tr, Xccr_tr, Xm_tr]).tocsr()
X_4v_un = hstack([Xw_un, Xcwb_un, Xccr_un, Xm_un]).tocsr()
X_4v_te = hstack([Xw_te, Xcwb_te, Xccr_te, Xm_te]).tocsr()

spec_keys = ['lr_4v', 'svc_4v', 'mask_lr', 'lgb_pure', 'cat_pure', 'xgb_pure', 'lgb_struct', 'cat_struct', 'xgb_struct']
oof_p = {k: np.zeros(len(train_df)) for k in spec_keys}
un_p = {k: np.zeros(len(unlabeled_df)) for k in spec_keys}
te_p = {k: np.zeros(len(test_df)) for k in spec_keys}

for fold, (t_idx, v_idx) in enumerate(skf.split(train_df, y_all)):
    y_t = y_all[t_idx]

    # 1. Sparse N-Gram Specialists
    m_lr = LogisticRegression(C=2.0, max_iter=1000, random_state=RANDOM_SEED + fold).fit(X_4v_tr[t_idx], y_t)
    oof_p['lr_4v'][v_idx] = m_lr.predict_proba(X_4v_tr[v_idx])[:, 1]
    un_p['lr_4v'] += m_lr.predict_proba(X_4v_un)[:, 1] / 5.0
    te_p['lr_4v'] += m_lr.predict_proba(X_4v_te)[:, 1] / 5.0

    m_svc = CalibratedClassifierCV(LinearSVC(C=0.5, random_state=RANDOM_SEED + fold), cv=3).fit(X_4v_tr[t_idx], y_t)
    oof_p['svc_4v'][v_idx] = m_svc.predict_proba(X_4v_tr[v_idx])[:, 1]
    un_p['svc_4v'] += m_svc.predict_proba(X_4v_un)[:, 1] / 5.0
    te_p['svc_4v'] += m_svc.predict_proba(X_4v_te)[:, 1] / 5.0

    m_mk = LogisticRegression(C=2.0, max_iter=1000, random_state=RANDOM_SEED + fold).fit(Xm_tr[t_idx], y_t)
    oof_p['mask_lr'][v_idx] = m_mk.predict_proba(Xm_tr[v_idx])[:, 1]
    un_p['mask_lr'] += m_mk.predict_proba(Xm_un)[:, 1] / 5.0
    te_p['mask_lr'] += m_mk.predict_proba(Xm_te)[:, 1] / 5.0

    # 2. Formatting-Free Pure+Info+LM+GenreZ GBDT Triad (116 features)
    X_p_tr_f = X_full_tr.iloc[t_idx][pure_116_cols]
    X_p_va_f = X_full_tr.iloc[v_idx][pure_116_cols]

    m_lgb_p = lgb.LGBMClassifier(n_estimators=450, learning_rate=0.04, num_leaves=31, min_child_samples=40, colsample_bytree=0.65, subsample=0.8, n_jobs=-1, random_state=RANDOM_SEED + fold, verbose=-1).fit(X_p_tr_f, y_t)
    oof_p['lgb_pure'][v_idx] = m_lgb_p.predict_proba(X_p_va_f)[:, 1]
    un_p['lgb_pure'] += m_lgb_p.predict_proba(X_full_un[pure_116_cols])[:, 1] / 5.0
    te_p['lgb_pure'] += m_lgb_p.predict_proba(X_full_te[pure_116_cols])[:, 1] / 5.0

    m_cat_p = CatBoostClassifier(iterations=850, learning_rate=0.05, depth=6, l2_leaf_reg=4.0, task_type='GPU', random_seed=RANDOM_SEED + fold, verbose=0).fit(X_p_tr_f, y_t)
    oof_p['cat_pure'][v_idx] = m_cat_p.predict_proba(X_p_va_f)[:, 1]
    un_p['cat_pure'] += m_cat_p.predict_proba(X_full_un[pure_116_cols])[:, 1] / 5.0
    te_p['cat_pure'] += m_cat_p.predict_proba(X_full_te[pure_116_cols])[:, 1] / 5.0

    m_xgb_p = xgb.XGBClassifier(n_estimators=450, learning_rate=0.04, max_depth=6, colsample_bytree=0.65, subsample=0.8, tree_method='hist', device='cuda', random_state=RANDOM_SEED + fold).fit(X_p_tr_f, y_t)
    oof_p['xgb_pure'][v_idx] = m_xgb_p.predict_proba(X_p_va_f)[:, 1]
    un_p['xgb_pure'] += m_xgb_p.predict_proba(X_full_un[pure_116_cols])[:, 1] / 5.0
    te_p['xgb_pure'] += m_xgb_p.predict_proba(X_full_te[pure_116_cols])[:, 1] / 5.0

    # 3. Damped-Structural + Info + LM + Stylo GBDT Triad (132 features)
    X_s_tr_f = X_full_tr.iloc[t_idx][all_132_cols]
    X_s_va_f = X_full_tr.iloc[v_idx][all_132_cols]

    m_lgb_s = lgb.LGBMClassifier(n_estimators=450, learning_rate=0.04, num_leaves=31, min_child_samples=50, colsample_bytree=0.45, subsample=0.8, reg_alpha=1.0, reg_lambda=2.0, n_jobs=-1, random_state=RANDOM_SEED + fold, verbose=-1).fit(X_s_tr_f, y_t)
    oof_p['lgb_struct'][v_idx] = m_lgb_s.predict_proba(X_s_va_f)[:, 1]
    un_p['lgb_struct'] += m_lgb_s.predict_proba(X_full_un[all_132_cols])[:, 1] / 5.0
    te_p['lgb_struct'] += m_lgb_s.predict_proba(X_full_te[all_132_cols])[:, 1] / 5.0

    m_cat_s = CatBoostClassifier(iterations=950, learning_rate=0.04, depth=6, l2_leaf_reg=5.0, task_type='GPU', random_seed=RANDOM_SEED + fold, verbose=0).fit(X_s_tr_f, y_t)
    oof_p['cat_struct'][v_idx] = m_cat_s.predict_proba(X_s_va_f)[:, 1]
    un_p['cat_struct'] += m_cat_s.predict_proba(X_full_un[all_132_cols])[:, 1] / 5.0
    te_p['cat_struct'] += m_cat_s.predict_proba(X_full_te[all_132_cols])[:, 1] / 5.0

    m_xgb_s = xgb.XGBClassifier(n_estimators=450, learning_rate=0.04, max_depth=5, colsample_bytree=0.45, subsample=0.8, reg_alpha=1.0, reg_lambda=2.0, tree_method='hist', device='cuda', random_state=RANDOM_SEED + fold).fit(X_s_tr_f, y_t)
    oof_p['xgb_struct'][v_idx] = m_xgb_s.predict_proba(X_s_va_f)[:, 1]
    un_p['xgb_struct'] += m_xgb_s.predict_proba(X_full_un[all_132_cols])[:, 1] / 5.0
    te_p['xgb_struct'] += m_xgb_s.predict_proba(X_full_te[all_132_cols])[:, 1] / 5.0

spec_table = []
for k in spec_keys:
    spec_table.append({
        'Specialist Model (10,000-Row 5-Fold CV)': k,
        '5-Fold OOF AUROC': round(roc_auc_score(y_all, oof_p[k]), 5),
        '5-Fold OOF Brier': round(brier_score_loss(y_all, oof_p[k]), 5)
    })
print(f"=== STAGE-1 9-SPECIALIST 5-FOLD OOF SUMMARY ({time.time()-t0:.1f}s) ===")
display(pd.DataFrame(spec_table))


=== STAGE-1 9-SPECIALIST 5-FOLD OOF SUMMARY (593.8s) ===


,"Specialist Model (10,000-Row 5-Fold CV)",5-Fold OOF AUROC,5-Fold OOF Brier
0,lr_4v,0.96448,0.07734
1,svc_4v,0.96559,0.06922
2,mask_lr,0.96531,0.09182
3,lgb_pure,0.98384,0.04588
4,cat_pure,0.98351,0.04566
5,xgb_pure,0.98344,0.04572
6,lgb_struct,0.99857,0.01102
7,cat_struct,0.99842,0.01088
8,xgb_struct,0.99847,0.01099


In [6]:
# 5. Stage-2 Domain-Balanced Transductive Self-Training Across unlabeled.csv AND test.csv (Adapting to Unseen Poetry & Novels!)
s1_b = cfg['stage1_blend']
def combine_9_specialists(p_dict):
    p_gbdt_pure = (p_dict['lgb_pure'] + p_dict['cat_pure'] + p_dict['xgb_pure']) / 3.0
    nw = s1_b['nofmt_weights']
    p_nofmt = nw.get('lr_4v', 0.22) * p_dict['lr_4v'] + nw.get('svc_4v', 0.22) * p_dict['svc_4v'] + nw.get('mask_lr', 0.14) * p_dict['mask_lr'] + nw.get('gbdt_pure', 0.42) * p_gbdt_pure
    l_nofmt = logit(np.clip(p_nofmt, 1e-5, 1 - 1e-5))
    sw = s1_b['struct_weights']
    l_struct = (
        sw.get('lgb_struct', 0.30) * logit(np.clip(p_dict['lgb_struct'], 1e-5, 1 - 1e-5)) +
        sw.get('cat_struct', 0.40) * logit(np.clip(p_dict['cat_struct'], 1e-5, 1 - 1e-5)) +
        sw.get('xgb_struct', 0.30) * logit(np.clip(p_dict['xgb_struct'], 1e-5, 1 - 1e-5))
    )
    fw = s1_b['fusion_weights']
    return expit(fw.get('nofmt', 0.62) * l_nofmt + fw.get('struct', 0.38) * l_struct), p_nofmt

oof_s1, oof_nofmt_s1 = combine_9_specialists(oof_p)
un_s1, _ = combine_9_specialists(un_p)
te_s1, te_nofmt_s1 = combine_9_specialists(te_p)

print(f"Stage-1 10,000-Row OOF AUROC (Formatting-Free Branch) : {roc_auc_score(y_all, oof_nofmt_s1):.5f}")
print(f"Stage-1 10,000-Row OOF AUROC (Full 9-Specialist Blend): {roc_auc_score(y_all, oof_s1):.5f}")

# Stage-2 Transductive Self-Training loaded from YAML
s2_cfg = cfg['stage2_transductive_self_training']
hc_un = (un_s1 >= s2_cfg['unlabeled_threshold_hi']) | (un_s1 <= s2_cfg['unlabeled_threshold_lo'])
y_un_pseudo = (un_s1[hc_un] >= 0.5).astype(int)

te_hc_indices = []
te_hc_labels = []
for dom, grp in test_df.groupby('domain'):
    dom_idx = grp.index.values
    dom_probs = te_s1[dom_idx]
    ai_cand = dom_idx[dom_probs >= s2_cfg['test_domain_threshold_ai']]
    hu_cand = dom_idx[dom_probs <= s2_cfg['test_domain_threshold_hu']]
    k_bal = min(len(ai_cand), len(hu_cand))
    if k_bal >= s2_cfg.get('min_balanced_k', 5):
        top_ai = ai_cand[np.argsort(te_s1[ai_cand])[::-1][:k_bal]]
        top_hu = hu_cand[np.argsort(te_s1[hu_cand])[:k_bal]]
        te_hc_indices.extend(top_ai)
        te_hc_labels.extend([1] * k_bal)
        te_hc_indices.extend(top_hu)
        te_hc_labels.extend([0] * k_bal)

te_hc_indices = np.array(te_hc_indices, dtype=int)
te_hc_labels = np.array(te_hc_labels, dtype=int)
print(f"Transductive Self-Training Pool -> Unlabeled HC: {hc_un.sum()} rows | Domain-Balanced Test HC: {len(te_hc_indices)} rows ({len(te_hc_indices)//2} AI, {len(te_hc_indices)//2} Human)")

X_4v_trans = hstack([X_4v_tr.T, X_4v_un[hc_un].T, X_4v_te[te_hc_indices].T]).T.tocsr()
y_trans = np.concatenate([y_all, y_un_pseudo, te_hc_labels])
w_trans = np.concatenate([np.ones(len(y_all)), np.full(len(y_un_pseudo), s2_cfg.get('pseudo_weight_unlabeled', 0.50)), np.full(len(te_hc_labels), s2_cfg.get('pseudo_weight_test', 0.35))])

m_lr_trans = LogisticRegression(**s2_cfg['lr_trans'], random_state=RANDOM_SEED).fit(X_4v_trans, y_trans, sample_weight=w_trans)
m_svc_trans = CalibratedClassifierCV(LinearSVC(C=s2_cfg['svc_trans']['C'], random_state=RANDOM_SEED), cv=s2_cfg['svc_trans']['cv']).fit(X_4v_trans, y_trans)

X_pure_trans = pd.concat([X_full_tr[pure_116_cols], X_full_un.loc[hc_un, pure_116_cols], X_full_te.iloc[te_hc_indices][pure_116_cols]], ignore_index=True)
X_struct_trans = pd.concat([X_full_tr[all_132_cols], X_full_un.loc[hc_un, all_132_cols], X_full_te.iloc[te_hc_indices][all_132_cols]], ignore_index=True)

m_lgb_p_trans = lgb.LGBMClassifier(**s2_cfg['lgb_p_trans'], random_state=RANDOM_SEED).fit(X_pure_trans, y_trans, sample_weight=w_trans)
m_cat_s_trans = CatBoostClassifier(**s2_cfg['cat_s_trans'], random_seed=RANDOM_SEED).fit(X_struct_trans, y_trans, sample_weight=w_trans)
m_lgb_s_trans = lgb.LGBMClassifier(**s2_cfg['lgb_s_trans'], random_state=RANDOM_SEED).fit(X_struct_trans, y_trans, sample_weight=w_trans)

s2_bw = s2_cfg['blend_weights']
p_trans_nofmt = (
    s2_bw['nofmt'].get('lr_trans', 0.30) * m_lr_trans.predict_proba(X_4v_te)[:, 1] +
    s2_bw['nofmt'].get('svc_trans', 0.30) * m_svc_trans.predict_proba(X_4v_te)[:, 1] +
    s2_bw['nofmt'].get('lgb_p_trans', 0.40) * m_lgb_p_trans.predict_proba(X_full_te[pure_116_cols])[:, 1]
)
l_trans_nofmt = logit(np.clip(p_trans_nofmt, 1e-5, 1 - 1e-5))
l_trans_struct = (
    s2_bw['struct'].get('cat_s_trans', 0.55) * logit(np.clip(m_cat_s_trans.predict_proba(X_full_te[all_132_cols])[:, 1], 1e-5, 1 - 1e-5)) +
    s2_bw['struct'].get('lgb_s_trans', 0.45) * logit(np.clip(m_lgb_s_trans.predict_proba(X_full_te[all_132_cols])[:, 1], 1e-5, 1 - 1e-5))
)
l_trans = s2_bw['fusion'].get('nofmt', 0.62) * l_trans_nofmt + s2_bw['fusion'].get('struct', 0.38) * l_trans_struct

# Final Submission v3 Logit Fusion
l_s1 = logit(np.clip(te_s1, 1e-5, 1 - 1e-5))
ff = cfg['final_fusion']
final_te_v3 = expit(ff.get('stage1_weight', 0.70) * l_s1 + ff.get('stage2_trans_weight', 0.30) * l_trans)

sub_v3 = pd.DataFrame({'id': test_df['id'].values, 'label': np.round(final_te_v3, 6)})
p_main = Path('submission.csv') if Path('feature_analysis.ipynb').exists() else Path('submissions/team-35/plum/submission.csv')
p_v3 = Path('submission_v3_grand_ensemble.csv') if Path('feature_analysis.ipynb').exists() else Path('submissions/team-35/plum/submission_v3_grand_ensemble.csv')
sub_v3.to_csv(p_main, index=False)
sub_v3.to_csv(p_v3, index=False)

print(f"Saved Submission v3 to '{p_main}' and '{p_v3}'!")
print(f"Submission v3 Summary -> Mean: {sub_v3['label'].mean():.4f}, Median: {sub_v3['label'].median():.4f}, Prop >= 0.5: {(sub_v3['label']>=0.5).mean():.4f}")
display(sub_v3.head(10))


Stage-1 10,000-Row OOF AUROC (Formatting-Free Branch) : 0.98503
Stage-1 10,000-Row OOF AUROC (Full 9-Specialist Blend): 0.99866
Transductive Self-Training Pool -> Unlabeled HC: 2188 rows | Domain-Balanced Test HC: 1702 rows (851 AI, 851 Human)
Saved Submission v3 to 'submission.csv' and 'submission_v3_grand_ensemble.csv'!
Submission v3 Summary -> Mean: 0.4531, Median: 0.1793, Prop >= 0.5: 0.4523


,id,label
0,13001,0.995896
1,13002,0.022183
2,13003,0.174755
3,13004,0.026447
4,13005,0.996280
5,13006,0.940904
6,13007,0.040435
7,13008,0.932781
8,13009,0.033062
9,13010,0.889070
